# 어휘 홀드아웃 평가 — 11차 레시피가 한 번도 안 본 단어에서 어떻게 하는가

**질문**: 11차 통제군 평가에서 단어 이득의 26%만 어휘 범위 확장에서 왔다(대장 「어휘
분해 결과」). 그 프록시는 118개(30종)뿐이고 통제군 쪽 사정에 좌우됐다. 이번엔
**11차 레시피 자체가** 한 번도 학습에서 못 본 단어(15종, `build_vocab_holdout.py`로
train·dev·test 전부에서 뺐다)에서 12차를 재고, 같은 단어에서 11차와 직접 비교한다.

**비교는 12차 대 11차뿐이다.** 7차재현·10차는 뺐다 — 15종 중 14종을 이미 알아서
공정한 건 `사탕`뿐이고 n이 너무 작다(대장 「12차 효율 조정」). 11차는 공정 행
457개만, 12차는 전 행 1640개를 잰다.

**판정 규칙(결과 보기 전에 대장에 적었다)**: 원래 dev·test였던 행(457개, 어느
모델도 그 클립은 학습에 안 씀)에서 12차의 단어 CER이 11차보다 0.10 이내로만
나쁘면 "어휘 몰라도 봐줄 만하다". 원래 train이었던 행(1183개)은 11차에게 이미
본 데이터라 비교에서 뺀다 — 12차 자신의 절대 성능에만 쓴다.

- 셀 1~3 · 6 · 7은 이전 노트북(`eval_adapters_big7_colab.ipynb`)의 셀을 **그대로**
  가져왔다. 셀 3의 ZIP만 `colab_trainset_big7_vochold.zip`으로 바뀌었다.
- 셀 4는 `ADAPTERS`를 `12차`·`11차`로 바꿨을 뿐 나머지는 같다.
- 판정에 **단어 단위 부트스트랩 95% CI**를 붙인다(행 457개가 단어 15종에 묶여 있다).
- 마지막 셀은 **학습량 교란**(12차는 학습 단어 클립도 12.6% 적다)을 아는 단어 dev에서
  확인한다. 새 추론은 없다.
- 준비물(드라이브): `colab_trainset_big7_vochold.zip` · `app_scorer.py` ·
  `whisper-608-lora-12차` · `-11차` · `whisper-608-control-big7-preds-11차.json`
  (통제군 평가 산출) · `whisper-608-dev-preds-12차.json`(학습 노트북 셀 8 산출).


## 셀 1 — GPU 확인 (Tesla T4가 떠야 함)

In [ ]:
!nvidia-smi

## 셀 2 — 설치 (환경 핀). numpy가 바뀔 때만 재시작을 요구한다

In [ ]:
# ══ 환경 고정 ═══════════════════════════════════════════════
# 회차 간 비교를 하려면 프레임워크가 회차마다 바뀌면 안 된다. 7차(8/18)는
# transformers 4.x, 7차재현(8/20)은 5.15.1이었고 그 사이에 같은 배치 설정이
# T4에서 OOM으로 죽었다. 올릴 때는 회차를 새로 시작하고 대장에 환경을 적는다.
TFV = "5.15.1"

# numpy는 Colab 이미지가 든 버전을 그대로 고정한다. 숫자를 하드코딩하면
# 이미지가 바뀔 때마다 다시 깨진다:
#   너무 높음 → AttributeError: module 'numpy' has no attribute '_no_nep50_warning'
#   너무 낮음 → ImportError: cannot import name '_center' from numpy._core.umath
import numpy, subprocess, sys
NPV = numpy.__version__
print("Colab 기본 numpy:", NPV, "→ 이 버전으로 고정 / transformers →", TFV)

# 핵심: numpy 핀을 같은 pip 호출 안에 넣어야 resolver가 도로 올리지 못한다.
!pip install -q -U "transformers=={TFV}" datasets peft accelerate evaluate jiwer librosa "numpy=={NPV}"
!pip uninstall -y -q torchao   # peft 버전검사와 충돌, LoRA엔 불필요

# **새 프로세스로** 확인한다. 이 커널이 이미 들고 있는 모듈은 pip이 디스크를
# 바꿔도 안 바뀌므로, 커널 안에서 import해 확인하면 거짓 통과한다.
_out = subprocess.run(
    [sys.executable, "-c",
     "import numpy, transformers; print(numpy.__version__, transformers.__version__)"],
    capture_output=True, text=True)
print("디스크:", (_out.stdout or _out.stderr).strip())
_disk = _out.stdout.split()

if len(_disk) != 2 or _disk[1] != TFV:
    raise SystemExit(f"transformers가 {TFV}가 아니다. 위 pip 출력을 확인해라.")
# numpy만이 재시작을 강제한다 — 이 커널이 이미 로드했기 때문이다.
# transformers·datasets 등은 아직 import 전이라 새 파일이 그대로 읽힌다.
if _disk[0] != NPV:
    raise SystemExit(f"numpy가 {NPV} → {_disk[0]}로 바뀌었다."
                     " [런타임 → 세션 다시 시작] 후 셀 2.5부터 다시 실행해라.")
print("✅ 설치 완료 — 재시작 없이 셀 2.5로 계속 진행")


## 셀 2.5 — 재시작 후 검증 (여기서 통과해야 아래가 의미 있음)
버전 3종을 찍고 실제 임포트까지 해본다. 실패하면 아래 셀들은 전부 같은 이유로 터지니 여기서 잡는다.
- `numpy has no attribute '_no_nep50_warning'` → 런타임 torch가 지금 numpy(≥2.3)를 못 따라감
- `cannot import name '_center' from numpy._core.umath` → 반대로 numpy가 너무 낮음

**처방은 둘 다 동일**: 런타임 → **세션 삭제 후 다시 시작** → 셀 2부터 다시(셀 2가 이미지 기준 numpy를 유지한다).
버전 숫자를 손으로 고쳐 맞추려 하지 말 것.

In [ ]:
import numpy, torch, transformers
print("numpy       :", numpy.__version__)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
if not transformers.__version__.startswith("5.15"):
    raise SystemExit("transformers 핀이 안 먹었다 — 셀 2를 다시 실행해라.")

# torch↔numpy 불일치면 바로 아래 줄에서 터진다(셀 4의 그 에러와 같은 원인)
from transformers import WhisperForConditionalGeneration
import librosa; librosa.stft(numpy.zeros(2048, dtype=numpy.float32))   # _center 계열 조기 탐지
print("✅ 임포트 정상 — 셀 3으로")

## 셀 3 — 드라이브 마운트 + vochold zip 풀기

In [ ]:
import os, torch, json
from google.colab import drive
drive.mount('/content/drive')

ZIP  = "/content/drive/MyDrive/colab_trainset_big7_vochold.zip"   # ← 업로드한 zip 경로
# big7_vochold = 12차·이 노트북 전용(build_vocab_holdout.py 산출). 단어 15종을
# train·dev·test 전부에서 빼고, 그 단어의 모든 클립을 holdout_eval.jsonl에 모았다.
# big7 = VS01 병합본(11차부터). train 2281→10485, dev 326→1427, test 417→2673 —
# **train만 4.6배가 아니라 dev·test도 커졌다.** 그래서 옛 dev/test 수치(0.3101 등)를
# 그대로 인용하면 안 된다 — 시험지 자체가 다른 시험지다(아래 셀 8 참고).
BASE = "/content/608-trainset"                       # 풀 위치(세션 로컬, 빠름)
if os.path.exists(ZIP):
    !unzip -oq "$ZIP" -d "$BASE"
else:
    # zip 대신 폴더째 올렸으면 그 경로를 BASE로 지정하고 위 unzip은 건너뛴다
    BASE = "/content/drive/MyDrive/608-trainset"

print("CUDA:", torch.cuda.is_available())

# dev는 선택이 아니다. dev 없이 돌리면 test로 체크포인트를 고르게 되고, 그러면
# 보고되는 CER이 홀드아웃 성능이 아니라 "시험지를 보며 고른 점수"가 된다.
# 4~6차 배치가 그 상태였다. 없으면 여기서 멈추고 재패키징을 요구한다.
if not os.path.exists(os.path.join(BASE, "holdout_eval.jsonl")):
    raise SystemExit(
        "holdout_eval.jsonl이 없다 — big7_vochold.zip이 아니라 big7.zip을 올린 것 아닌지 확인해라."
    )

for f in ["train.jsonl","dev.jsonl","test.jsonl","holdout_eval.jsonl"]:
    p=os.path.join(BASE,f); n=sum(1 for _ in open(p,encoding="utf-8"))
    print(f"  {f}: {n}개")
print("wav:", sum(len(fs) for _,_,fs in os.walk(os.path.join(BASE,'wav'))), "개")

## 셀 4 — 모델 + 어댑터 로드 (12차 포함)

In [ ]:
# 할당자가 크기별로 굳은 세그먼트를 재사용하지 못해 여유가 있는데도 OOM이 나는
# 경우가 있다(생성 평가는 배치마다 시퀀스 길이가 달라 특히 그렇다). 이 설정은
# **CUDA 컨텍스트가 만들어지기 전에** 걸려야 효과가 있어 맨 위에 둔다.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

# 베이스 모델. zero-shot 비교(셀 10) 결과로 large-v3-turbo를 채택했다.
#
#   모델              단어 CER   문장 CER
#   small               2.000      0.404
#   medium              1.376      0.363
#   large-v3-turbo      0.890      0.321   ← 두 지표 모두 승
#
# 단어가 모델 크기에 훨씬 민감하다(2.2배 개선 vs 문장 1.26배). 앱 최약점이
# 단어 수준 구음장애라 이 차이가 그대로 제품 성능이다. turbo는 디코더가 얕아
# large-v3(1550M)보다 훨씬 빠르면서 T4에 들어간다.
#
# 되돌리려면 이 한 줄만 바꾼다. 프로세서도 같은 ID로 로드되므로 mel 빈 수
# (large-v3 계열 128 vs small/medium 80)가 자동으로 맞는다 — 프로세서를
# 재사용하면 입력이 조용히 깨진다.
MODEL_ID = "openai/whisper-large-v3-turbo"

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="ko", task="transcribe")
# **fp32로 명시 로드해야 한다.** large-v3-turbo는 config에 torch_dtype: float16이
# 박혀 있어 최신 transformers가 fp16 가중치로 올린다. 그러면 특징 추출기가 내놓는
# fp32 입력과 어긋나 인코더 conv에서 터진다:
#   RuntimeError: Input type (float) and bias type (c10::Half) should be the same
# small은 config가 fp32라 이 문제가 없어 지금까지 안 드러났다.
#
# 혼합정밀은 Trainer의 fp16=True(autocast)가 맡는다. 가중치까지 fp16으로 두고
# 학습하면 마스터 가중치가 없어 불안정하다. T4는 Turing이라 bf16이 없어 이 조합이
# 유일한 선택지다.
# **attn_implementation을 명시한다.** 기본값에 맡기면 transformers 버전에 따라
# eager로 떨어질 수 있고, 그러면 어텐션 확률 행렬이 통째로 저장된다:
#   배치4 x 헤드20 x 1500 x 1500 x 2B = 360MB/층  x  인코더 32층 = 11.5GB
# sdpa는 그걸 아예 만들지 않는다. 이 한 줄이 T4에 들어가느냐 마느냐를 가른다.
# (셀 2가 transformers를 핀 없이 -U로 깔기 때문에 기본값에 기댈 수 없다.)
try:
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32, attn_implementation='sdpa')
except (ValueError, ImportError) as e:
    print('sdpa 거부됨:', e)
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
print('어텐션 구현:', getattr(model.config, '_attn_implementation', '?'),  # 'sdpa'여야 한다
      '| torch', torch.__version__)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

# ══ 어댑터 로드 — 학습된 LoRA를 붙인다(새로 만들지 않는다) ═══════════════
# 원본 셀 4는 여기서 get_peft_model(LoraConfig(...))로 **새** LoRA를 만든다.
# 통제군 평가는 학습을 안 하므로 저장된 어댑터를 읽는다. 어댑터 여러 개를 한
# 모델에 얹어 set_adapter로 바꿔 가며 잰다(베이스 3.2GB를 한 번만 올린다).
from peft import PeftModel

# 평가 순서. 11차를 먼저 재는 이유: 학습 중 기록된 dev 값과 이 노트북의 값이
# 맞는지(=평가 경로가 같은지) 가장 먼저 확인해야 나머지 숫자를 믿을 수 있다.
ADAPTERS = ["12차", "11차"]   # 둘 다 필수. 7차재현·10차는 뺐다(대장 「12차 효율 조정」)
ADAPTER_DIR = "/content/drive/MyDrive/whisper-608-lora-{}"

found, absent = [], []
for _n in ADAPTERS:
    (found if os.path.isdir(ADAPTER_DIR.format(_n)) else absent).append(_n)
print("어댑터 있음:", found)
if absent:
    print("어댑터 없음:", absent, "→ 이 회차는 건너뛴다(재학습 없이는 통제군이 못 된다)")
if absent:
    raise SystemExit(f"필수 어댑터가 없다: {absent} — whisper-608-lora-<회차> 폴더를 확인해라.")

model = PeftModel.from_pretrained(model, ADAPTER_DIR.format(found[0]), adapter_name=found[0])
for _n in found[1:]:
    model.load_adapter(ADAPTER_DIR.format(_n), adapter_name=_n)
model.eval()
print("로드 완료. 활성:", model.active_adapter)

# 생성(평가) 설정 — 이거 없으면 whisper가 반복 환각으로 CER이 2.0+로 부풀려진다.
model.generation_config.language = "ko"
model.generation_config.task = "transcribe"
model.generation_config.no_repeat_ngram_size = 3   # 같은 구절 반복 억제

# ── 평가 시간의 대부분은 여기서 결정된다 ─────────────────────────
# 자기회귀 생성이라 **배치 안에서 가장 긴 시퀀스**가 그 배치의 시간을 정한다.
# 그리고 학습 전/초기 모델은 환각으로 거의 항상 상한까지 뱉는다 — 즉 상한값이
# 곧 평가 비용이다.
#
# 실측 라벨 길이 (colab_trainset_big6):
#     train 단어  중앙 1자 · 최대  2자
#     train 문장  중앙 13자 · p95 34자 · 최대 93자
#     dev  문장  최대 56자   |   test 문장 최대 44자
#
# 100토큰은 dev/test 최장(56자)의 두 배 가까이라 낭비다. 72로 낮춰 환각 꼬리를
# 자른다. 잘림 위험은 없다 — 한국어는 음절당 1토큰 안팎이라 56자면 60토큰 미만이다.
model.generation_config.max_new_tokens = 72

# **KV 캐시를 반드시 켠다.** 위에서 config.use_cache=False로 둔 건 학습용 관례인데,
# 그게 생성까지 따라가면 매 토큰마다 전체 시퀀스를 재계산해 평가가 몇 배 느려진다.
# (gradient_checkpointing을 안 쓰므로 학습에도 use_cache=False가 필요 없지만,
#  건드리면 학습 조건이 바뀌므로 생성 쪽만 명시적으로 켠다.)
model.generation_config.use_cache = True


## 셀 5a — 데이터 (holdout_eval만)

In [ ]:
import collections
from datasets import load_dataset, Audio

# holdout_eval.jsonl만 본다 — build_vocab_holdout.py가 만든, 15종의 모든 클립을
# 원래 스플릿(orig_split)과 함께 모아 둔 파일이다. train/dev/test는 이 실험에서
# 체크포인트 선택용으로만 쓰이고(셀 8), 여기서는 다시 안 읽는다.
if not os.path.exists(f"{BASE}/holdout_eval.jsonl"):
    raise SystemExit(
        "holdout_eval.jsonl이 없다 — big7 zip을 잘못 올렸을 수 있다. "
        "colab_trainset_big7_vochold.zip인지 확인해라."
    )

raw = [json.loads(l) for l in open(f"{BASE}/holdout_eval.jsonl", encoding="utf-8")]
words = [r["text"].strip() for r in raw]
orig_split = [r["orig_split"] for r in raw]

ds = load_dataset("json", data_files={"holdout": f"{BASE}/holdout_eval.jsonl"})
ds = ds.map(lambda b: {"audio": os.path.join(BASE, b["audio"])})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))


def prep(b):   # eval_adapters_big7_colab.ipynb의 prep과 글자 그대로 같다
    a = b["audio"]
    b["input_features"] = processor.feature_extractor(
        a["array"], sampling_rate=16000).input_features[0]
    b["labels"] = processor.tokenizer(b["text"]).input_ids
    return b


ds = ds.map(prep, remove_columns=ds["holdout"].column_names, writer_batch_size=20)

assert len(ds["holdout"]) == len(words) == len(orig_split), "행 수가 안 맞는다"
print(ds)
print("단어 종류", len(set(words)), "개 ·", dict(collections.Counter(orig_split)))


## 셀 6 — 데이터 콜레이터 (whisper 필수)

In [ ]:
from dataclasses import dataclass
from typing import Any
@dataclass
class Collator:
    processor: Any
    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        labels = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        lab = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (lab[:,0] == self.processor.tokenizer.bos_token_id).all():
            lab = lab[:,1:]
        batch["labels"] = lab
        return batch
collator = Collator(processor)

## 셀 7 — CER 지표 로드

In [ ]:
# 원본(학습 노트북 셀 7)의 compute_metrics는 dev_task_types·test_task_types를 참조한다.
# 이 노트북에는 그 변수가 없어 그대로 가져오면 NameError다(#241에서 그렇게 들어갔다).
# 판정은 아래 셀 5가 evaluate로 직접 계산하므로 Trainer에 지표 함수를 넘기지 않는다.
import evaluate
cer_metric = evaluate.load("cer")


## 셀 7.5 — 앱 채점기 로드 (제품과 같은 잣대)

608 평가의 표준 CER은 **앱 조건보다 비관적**이다. 앱은 단어 과제에 음소 유사 가중
거리를 써서 조음 유사 혼동(ㅂ↔ㅍ, 종성 탈락)을 부분 오류로 본다. 같은 채점기로
재야 "이 모델을 붙이면 앱이 얼마나 좋아지나"에 답할 수 있다.

무엇보다 앱이 내는 판정은 오류율이 아니라 **임계값 통과 여부**(≤0.34)다.
평균이 경계 아래인 것과 항목별 통과율은 다른 이야기다.


In [ ]:
# scripts/asr_eval/app_scorer.py 를 Colab으로 가져온다.
# 리포를 통째로 올리기 번거로우면 이 셀에 붙여넣어도 되지만, 원본이 바뀌면
# 어긋나므로 파일을 올리는 쪽을 권한다.
#
# 원본(단일 진실): frontend/src/memory-link/patient/quiz/domain/phoneticDistance.ts
#                  frontend/src/memory-link/patient/quiz/domain/speechScore.ts
# 파이썬 이식본: scripts/asr_eval/app_scorer.py (test_app_scorer.py가 두 구현을 고정)

import os, sys, urllib.request

APP_SCORER = "/content/app_scorer.py"
if not os.path.exists(APP_SCORER):
    # (a) 드라이브에 올려 뒀으면 그걸 쓴다
    cand = "/content/drive/MyDrive/app_scorer.py"
    if os.path.exists(cand):
        import shutil; shutil.copy(cand, APP_SCORER)
    else:
        raise SystemExit(
            "app_scorer.py가 없다. scripts/asr_eval/app_scorer.py를 드라이브"
            " 루트에 올리거나 /content/에 업로드하라."
        )
sys.path.insert(0, "/content")
import app_scorer
print("앱 채점기 로드 · 정답 임계값:", app_scorer.SPEECH_PASS_THRESHOLD)


## 셀 5 — 어댑터별 홀드아웃 평가 + 판정

In [ ]:
import json, time, unicodedata, re, random, warnings, gc
import transformers as _tf
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
_tf.logging.set_verbosity_error(); warnings.filterwarnings("ignore")

FAIR_SPLITS = {"dev", "test"}   # 원래 이 스플릿이었던 행만 "어느 모델도 못 본 클립"이다

# 어댑터별로 잴 행. 11차는 원래 train 행(1183개)을 이미 학습했으니 그 행은
# 비교에 못 쓴다 — 공정 행만 잰다. 12차는 전 행(절대 성능 + 공정 비교)을 잰다.
# 7차재현·10차는 뺐다: 15종 중 14종을 이미 알아 공정한 건 사탕뿐이고 n이 너무
# 작다(대장 「12차 효율 조정」).
SCOPE = {"12차": "all", "11차": "fair"}

RESULT_PATH = "/content/drive/MyDrive/whisper-608-vochold-results.json"
PRED_PATH = "/content/drive/MyDrive/whisper-608-vochold-preds-{}.json"

# 사전 등록한 판정 문턱(대장 「12차 계획」, 결과 보기 전에 적었다)
FAIR_TOLERANCE = 0.10   # 원래 dev·test였던 행에서 12차가 11차보다 이 이상 나쁘면 안 된다
BOOT_B, BOOT_SEED = 1000, 42


def _norm(t):
    t = unicodedata.normalize("NFC", t or "")
    t = re.sub(r"[^\w가-힣\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


try:
    import evaluate
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "evaluate", "jiwer"], check=True)
    import evaluate
_cer_metric = evaluate.load("cer")


def _cer(hyps, refs):
    if not refs:
        return float("nan")
    return _cer_metric.compute(predictions=hyps, references=refs)


def grp(hyps, refs):
    """한 묶음의 CER · 정확 일치율 · 앱 잣대 통과율."""
    if not refs:
        return None
    return {"n": len(refs), "cer": _cer(hyps, refs),
            "acc": sum(_norm(h) == _norm(r) for h, r in zip(hyps, refs)) / len(refs),
            "app_pass": app_scorer.score_batch(hyps, refs, "word")["pass_rate"]}


def summarize(preds):
    """preds = [{row, word, orig_split, ref, hyp}] — 이 어댑터가 잰 행만."""
    def sub(cond):
        rs = [p for p in preds if cond(p)]
        return grp([p["hyp"] for p in rs], [p["ref"] for p in rs])
    fair = lambda p: p["orig_split"] in FAIR_SPLITS
    out = {"fair": sub(fair),
           "all": sub(lambda p: True) if any(not fair(p) for p in preds) else None}
    out["by_word"] = {w: sub(lambda p, w=w: fair(p) and p["word"] == w)
                      for w in sorted({p["word"] for p in preds})}
    return out


def verdict(new_preds, ctl_preds):
    """12차 대 11차 — 공정 행에서, 사전 등록한 문턱으로. 단어 단위 부트스트랩 CI를 붙인다.

    행 457개는 단어 15종에 묶여 있어 실효 표본은 15에 가깝다. 행을 재표집하면 CI가
    지나치게 좁아진다 — 그래서 **단어를** 복원 추출한다.
    """
    ctl = {p["row"]: p for p in ctl_preds if p["orig_split"] in FAIR_SPLITS}
    pairs = [(p, ctl[p["row"]]) for p in new_preds if p["row"] in ctl]
    assert len(pairs) == len(ctl), f"공정 행이 안 맞는다: 12차 {len(pairs)} vs 11차 {len(ctl)}"
    assert all(a["ref"] == b["ref"] for a, b in pairs), "같은 행의 정답이 다르다 — 행 번호가 어긋났다"

    def diff(ps):
        return (_cer([a["hyp"] for a, _ in ps], [a["ref"] for a, _ in ps])
                - _cer([b["hyp"] for _, b in ps], [b["ref"] for _, b in ps]))

    d = diff(pairs)   # 양수 = 12차가 나쁘다
    by_w = {}
    for a, b in pairs:
        by_w.setdefault(a["word"], []).append((a, b))
    ws = sorted(by_w)
    rng = random.Random(BOOT_SEED)
    boots = sorted(diff([x for w in (rng.choice(ws) for _ in ws) for x in by_w[w]])
                   for _ in range(BOOT_B))
    lo, hi = boots[int(0.025 * BOOT_B)], boots[int(0.975 * BOOT_B) - 1]

    if d <= FAIR_TOLERANCE:
        v = f"어휘 몰라도 봐줄 만하다 (대가 {d:+.4f} <= 문턱 {FAIR_TOLERANCE})"
    else:
        v = f"어휘 노출이 이득의 대부분이다 (대가 {d:+.4f} > 문턱 {FAIR_TOLERANCE})"
    if lo <= FAIR_TOLERANCE <= hi:
        v += f" — 단, 95% CI [{lo:+.4f}, {hi:+.4f}]가 문턱을 가로지른다: 확정적이지 않다"
    else:
        v += f" · 95% CI [{lo:+.4f}, {hi:+.4f}]"
    return v, d, (lo, hi)


def load_preds(name):
    return json.load(open(PRED_PATH.format(name), encoding="utf-8"))


args = Seq2SeqTrainingArguments(
    output_dir="/content/vochold_tmp",
    per_device_eval_batch_size=8, dataloader_num_workers=2, fp16=True,
    predict_with_generate=True, remove_unused_columns=False,
    label_names=["labels"], report_to="none")

fair_rows = [i for i, s in enumerate(orig_split) if s in FAIR_SPLITS]
all_rows = list(range(len(words)))
print(f"행: 전체 {len(all_rows)} · 공정(원래 dev+test) {len(fair_rows)}")

results = json.load(open(RESULT_PATH, encoding="utf-8")) if os.path.exists(RESULT_PATH) else {}
for name in found:
    if name not in SCOPE:
        continue
    if name in results and os.path.exists(PRED_PATH.format(name)):
        print(f"[{name}] 이미 잰 결과가 있다 — 건너뜀")
        continue
    rows = all_rows if SCOPE[name] == "all" else fair_rows
    model.set_adapter(name)
    assert model.active_adapter == name, f"어댑터가 안 바뀌었다: {model.active_adapter}"
    trainer = Seq2SeqTrainer(model=model, args=args, data_collator=collator,
                             processing_class=processor)
    t0 = time.time()
    pred = trainer.predict(ds["holdout"].select(rows))
    lab = pred.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    hyps = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
    refs = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    preds = [{"row": i, "word": words[i], "orig_split": orig_split[i], "ref": r, "hyp": h}
             for i, r, h in zip(rows, refs, hyps)]
    # 예측을 먼저 저장한다 — 목표 단어 검증(TODOS 1번)이 이 자유 받아쓰기를 다시 쓴다
    json.dump(preds, open(PRED_PATH.format(name), "w", encoding="utf-8"), ensure_ascii=False)
    res = summarize(preds)
    res["scope"] = SCOPE[name]
    results[name] = res
    json.dump(results, open(RESULT_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    fair = res["fair"]
    print(f"[{name}] {(time.time()-t0)/60:.1f}분 · {len(rows)}행 · 공정 CER {fair['cer']:.4f}"
          f" · 정확도 {fair['acc']:.1%} · 앱 통과 {fair['app_pass']:.1%} (n={fair['n']})", flush=True)
    del trainer
    gc.collect(); torch.cuda.empty_cache()

# ══ 요약 ═════════════════════════════════════════════════════
print()
print("  " + "어댑터".ljust(8) + "범위".rjust(6) + "공정 CER".rjust(10) + "정확도".rjust(8)
      + "앱 통과".rjust(8) + "전체 CER(참고)".rjust(16))
for name, r in results.items():
    f, a = r["fair"], r.get("all")
    print("  " + name.ljust(8) + r.get("scope", "?").rjust(6) + f"{f['cer']:.4f}".rjust(10)
          + f"{f['acc']:.1%}".rjust(8) + f"{f['app_pass']:.1%}".rjust(8)
          + (f"{a['cer']:.4f} (n={a['n']})" if a else "-").rjust(16))
print("  ※ 앱 통과율 = 퇴역한 문자열 채점기(app_scorer.py). 지금 앱은 Azure로 채점한다.")
print("    근접 단어를 봐줘서 부풀려져 있다(대장 「foil 실측과 문자열 채점기의 성질」).")

if "12차" in results and "11차" in results:
    v, d, ci = verdict(load_preds("12차"), load_preds("11차"))
    print(f"\n[판정] 12차 대 11차, 공정 행 {len(fair_rows)}개 → {v}")
    results["verdict"] = {"text": v, "cost": d, "ci95": list(ci), "boot_b": BOOT_B}
    json.dump(results, open(RESULT_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)


    print("\n[단어별 CER — 공정 행]")
    print("  " + "단어".ljust(6) + "12차".rjust(10) + "11차".rjust(10) + "대가".rjust(10) + "n".rjust(6))
    for w in sorted(results["12차"]["by_word"]):
        a, b = results["12차"]["by_word"].get(w), results["11차"]["by_word"].get(w)
        if not a or not b:
            continue
        print("  " + w.ljust(6) + f"{a['cer']:.3f}".rjust(10) + f"{b['cer']:.3f}".rjust(10)
              + f"{a['cer'] - b['cer']:+.3f}".rjust(10) + str(a["n"]).rjust(6))


## 셀 6 — 학습량 교란 확인 (아는 단어 dev, 새 추론 없음)

In [ ]:
# 12차는 단어 15종만 빠진 게 아니라 학습 단어 클립도 12.6%(1183개) 줄었다. 그래서
# 위 대가에는 "어휘를 몰라서"와 "덜 배워서"가 섞일 수 있다. 둘 다 **아는 단어**인
# dev 행에서 12차와 11차를 비교하면 "덜 배워서" 몫이 나온다. 새 추론은 없다 —
# 11차는 통제군 평가가 남긴 big7 dev 예측을, 12차는 학습 노트북 셀 8이 남긴 dev
# 예측을 읽는다.
#
# 사전 등록(대장 「12차 효율 조정」): 아는 단어 dev CER 차이(12차 − 11차)가
#   <= 0.02 → 학습량 교란은 무시할 수 있다(11차 판정의 문턱과 같은 값)
#   >  0.02 → 교란 있음. 위 대가와 함께 "대가 − 이 차이"를 민감도로 같이 적는다
CONFOUND_TOL = 0.02
P11 = "/content/drive/MyDrive/whisper-608-control-big7-preds-11차.json"
P12 = "/content/drive/MyDrive/whisper-608-dev-preds-12차.json"

missing = [p for p in (P11, P12) if not os.path.exists(p)]
if missing:
    print("교란 확인 건너뜀 — 예측 파일이 없다:", missing)
    print("  11차: eval_adapters_big7_colab.ipynb가 만든다 / 12차: 학습 노트북 셀 8이 만든다")
else:
    holdout_words = set(json.load(open(f"{BASE}/holdout_words.json", encoding="utf-8")))
    p11 = json.load(open(P11, encoding="utf-8"))
    p12 = json.load(open(P12, encoding="utf-8"))
    # build_vocab_holdout.py와 같은 기준으로 빼면 행 순서가 12차 dev와 같아진다
    keep11 = [r for r in p11 if not (r["task"] == "wordlist" and r["ref"].strip() in holdout_words)]
    assert len(keep11) == len(p12), f"행 수가 다르다: 11차 {len(keep11)} vs 12차 {len(p12)}"
    bad = sum(a["ref"].strip() != b["ref"].strip() for a, b in zip(keep11, p12))
    assert bad == 0, f"같은 위치의 정답이 {bad}곳 다르다 — 행 정렬이 어긋났다"

    print(f"아는 단어 dev — 11차 {len(p11)}행에서 홀드아웃 {len(p11) - len(keep11)}행을 빼고 {len(keep11)}행 정렬 확인")
    conf = {}
    for task, label in (("wordlist", "단어"), ("narrative", "문장")):
        idx = [i for i, r in enumerate(keep11) if r["task"] == task]
        c11 = _cer([keep11[i]["hyp"] for i in idx], [keep11[i]["ref"] for i in idx])
        c12 = _cer([p12[i]["hyp"] for i in idx], [p12[i]["ref"] for i in idx])
        conf[task] = c12 - c11
        print(f"  {label}: 11차 {c11:.4f} · 12차 {c12:.4f} · 차이 {c12 - c11:+.4f} (n={len(idx)})")

    gap = conf["wordlist"]
    if gap <= CONFOUND_TOL:
        msg = f"학습량 교란은 무시할 수 있다 (아는 단어 차이 {gap:+.4f} <= {CONFOUND_TOL})"
    else:
        msg = f"교란 있음 (아는 단어 차이 {gap:+.4f} > {CONFOUND_TOL})"
        if "verdict" in results:
            msg += f" — 민감도: 대가 {results['verdict']['cost']:+.4f} − {gap:+.4f} = {results['verdict']['cost'] - gap:+.4f}"
    print(f"\n[교란 확인] {msg}")
    results["confound"] = {"word_gap": gap, "sent_gap": conf["narrative"], "text": msg}
    json.dump(results, open(RESULT_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
